2 — Training (export ONNX)\nEntraîne le modèle final sur **tout** `dataset.parquet` et l'exporte en **ONNX** (portable : Python, C#, MQL5/MT5 natif).\n\n⚠️ À ne lancer **qu'après** le verdict **GO** du notebook 3 (validation).

In [27]:
# pip install scikit-learn skl2onnx onnxruntime joblib pandas numpy pyarrow

In [28]:
# === CONFIG PARTAGEE ===
SYMBOLS  = ['EURJPY','USDJPY','GBPJPY','GBPUSD','EURUSD','NZDUSD','AUDJPY','AUDUSD']
SL_PIPS  = {'AUDJPY':38,'AUDUSD':28,'EURJPY':48,'EURUSD':32,
            'GBPJPY':59,'GBPUSD':42,'NZDUSD':28,'USDJPY':39}
SPREAD_PIPS = {'AUDJPY':0.7,'AUDUSD':0.4,'EURJPY':0.6,'EURUSD':0.2,
               'GBPJPY':0.8,'GBPUSD':0.4,'NZDUSD':0.5,'USDJPY':0.3}
RR=1.70; MAX_HOLD_H=168; YEARS_BACK=10; SELECT_Q=0.95
FEAT=['rsi14','pctB','bw','ext','adx14','atr_norm','body','up_wick','dn_wick','bull',
      'dist_hi20','dist_lo20','bars_since_flip','ret5_atr','d1_state','d1_rsi',
      'd1_aligned','hour','dow','dir_b','sym_c']
SYM_CODE={s:i for i,s in enumerate(sorted(SYMBOLS))}
DATASET='dataset.parquet'


In [29]:
# === ENTRAINEMENT + SEUIL HORS ECHANTILLON ===
import pandas as pd, numpy as np, json
from sklearn.ensemble import HistGradientBoostingRegressor

d=pd.read_parquet(DATASET).dropna(subset=['R_net'])
d['bars_since_flip']=d['bars_since_flip'].clip(upper=300)  # parite EA (garde-fou si vieux dataset)
d['year']=pd.to_datetime(d.time).dt.year
d=d.sort_values('time').reset_index(drop=True)
print(len(d),'lignes |',d.year.min(),'->',d.year.max())

def make_model():
    return HistGradientBoostingRegressor(max_iter=300,max_depth=6,learning_rate=0.06,
        l2_regularization=1.0,random_state=0)

# --- 1) SEUIL = q95 des predictions HORS ECHANTILLON (walk-forward annuel) ---
# Un quantile pris sur les predictions in-sample (l'ancien code) surestime la
# selectivite : le modele connait ses propres cibles -> predictions plus hautes
# sur son train que sur des donnees nouvelles -> en live, moins de signaux que
# prevu et un seuil qui ne correspond plus au quantile vise.
oos=[]
for y in sorted(d.year.unique())[-5:]:
    tr=d[d.year<y]
    if len(tr)<20000: continue
    m=make_model(); m.fit(tr[FEAT],tr['R_net'])
    oos.append(m.predict(d[d.year==y][FEAT]))
    print(f'  fold {y}: train<{y} ({len(tr)} l.) -> {len(oos[-1])} preds OOS')
assert oos, "pas assez d'annees pour calibrer le seuil OOS"
threshold=float(np.quantile(np.concatenate(oos),SELECT_Q))
print(f'seuil OOS (q{SELECT_Q}): {threshold:+.4f}')

# --- 2) modele final entraine sur TOUT le dataset ---
model=make_model()
model.fit(d[FEAT],d['R_net'])
thr_ins=float(np.quantile(model.predict(d[FEAT]),SELECT_Q))
print(f'(info : seuil in-sample {thr_ins:+.4f} — NE PAS utiliser, biais de selectivite)')

317134 lignes | 2001 -> 2026
  fold 2022: train<2022 (260806 l.) -> 12432 preds OOS
  fold 2023: train<2023 (273238 l.) -> 12432 preds OOS
  fold 2024: train<2024 (285670 l.) -> 12480 preds OOS
  fold 2025: train<2025 (298150 l.) -> 12432 preds OOS
  fold 2026: train<2026 (310582 l.) -> 6552 preds OOS
seuil OOS (q0.95): +0.1433
(info : seuil in-sample +0.1267 — NE PAS utiliser, biais de selectivite)


In [30]:
# === EXPORT ONNX ===
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

onx=convert_sklearn(model, initial_types=[('input',FloatTensorType([None,len(FEAT)]))],
                    target_opset=15)
with open('model.onnx','wb') as f: f.write(onx.SerializeToString())
json.dump({'threshold':threshold,'features':FEAT,'select_q':SELECT_Q,
           'n_train':len(d)},open('model_meta.json','w'),indent=1)
print('-> model.onnx + model_meta.json')

-> model.onnx + model_meta.json


In [32]:
# === PARITE v3 : diagnostic + parite DECISIONNELLE (criteres en taux) ===
import onnxruntime as ort, numpy as np

sess=ort.InferenceSession('model.onnx')
X64=d[FEAT].values[:20000]
X32=X64.astype('float32')

p_onnx=sess.run(None,{'input':X32})[0].ravel()
p_skl64=model.predict(d[FEAT].iloc[:20000])
p_skl32=model.predict(pd.DataFrame(X32,columns=FEAT))

print(f'ecart sklearn64 vs ONNX      : {np.abs(p_onnx-p_skl64).max():.6f}')
print(f'ecart sklearn64 vs sklearn32 : {np.abs(p_skl32-p_skl64).max():.6f}  <- float32 des entrees')
print(f'ecart sklearn32 vs ONNX      : {np.abs(p_onnx-p_skl32).max():.6f}  <- conversion')

dec_skl = p_skl64>=threshold
dec_onx = p_onnx >=threshold
m_dis   = dec_skl!=dec_onx
disagree= int(m_dis.sum())
print(f'\ndesaccords de decision au seuil {threshold:+.4f}: {disagree}/{len(p_onnx)} ({disagree/len(p_onnx):.4%})')
if disagree:
    print(np.c_[p_skl64[m_dis][:10], p_onnx[m_dis][:10]])

# criteres EN TAUX : desaccords totaux <0.1% ET desaccords loin du seuil <0.02%
far = m_dis & (np.abs(p_skl64-threshold)>=0.01)
rate_dis, rate_far = disagree/len(p_onnx), far.sum()/len(p_onnx)
print(f'desaccords loin du seuil (>=0.01): {int(far.sum())} ({rate_far:.4%})')
assert rate_dis<0.001 and rate_far<0.0002, 'PARITE DECISIONNELLE NON RESPECTEE'
print('\nPARITE DECISIONNELLE OK — model.onnx deployable')
print('Note: ecarts sur lignes-frontiere = precision float32 des arbres, attendu.')

ecart sklearn64 vs ONNX      : 0.053994
ecart sklearn64 vs sklearn32 : 0.079788  <- float32 des entrees
ecart sklearn32 vs ONNX      : 0.079788  <- conversion

desaccords de decision au seuil +0.1433: 2/20000 (0.0100%)
[[0.16727782 0.11328365]
 [0.14622245 0.14198847]]
desaccords loin du seuil (>=0.01): 1 (0.0050%)

PARITE DECISIONNELLE OK — model.onnx deployable
Note: ecarts sur lignes-frontiere = precision float32 des arbres, attendu.


In [33]:
# === FINALISATION : parite decisionnelle au seuil OOS + model_meta.json ===
# (l'ancienne version recalculait ici un seuil q95 sur les predictions ONNX
#  IN-SAMPLE et l'ecrivait dans model_meta.json — c'est le seuil OOS de la
#  cellule d'entrainement qui doit etre fige, pas un quantile in-sample)
import json
Xall = d[FEAT].astype('float32').values
p_onnx_all = sess.run(None, {'input': Xall})[0].ravel()
p_skl_all  = model.predict(d[FEAT])

dis = ((p_skl_all >= threshold) != (p_onnx_all >= threshold)).sum()
print(f'desaccords de decision au seuil OOS {threshold:+.4f}: {dis}/{len(d)} ({dis/len(d):.5%})')
assert dis/len(d) < 0.001, 'PARITE DECISIONNELLE NON RESPECTEE'

sel_rate = (p_onnx_all >= threshold).mean()
print(f'taux de selection in-sample a ce seuil: {sel_rate:.2%} '
      f'(>{(1-SELECT_Q):.0%} attendu : normal, le modele score plus haut sur son train)')

json.dump({'threshold': threshold, 'features': FEAT, 'select_q': SELECT_Q,
           'n_train': len(d), 'threshold_method': 'oos-walk-forward-5y',
           'bars_since_flip_clip': 300, 'runtime': 'onnx-float32'},
          open('model_meta.json', 'w'), indent=1)
print('model_meta.json fige avec le seuil OOS -> reporter CE seuil dans InpThreshold de l EA.')

desaccords de decision au seuil OOS +0.1433: 2/317134 (0.00063%)
taux de selection in-sample a ce seuil: 4.09% (>5% attendu : normal, le modele score plus haut sur son train)
model_meta.json fige avec le seuil OOS -> reporter CE seuil dans InpThreshold de l EA.
